In [11]:
from ome_zarr_models.v05.plate import Plate, Column, Row, Acquisition, WellInPlate
from ome_zarr_models.common.well_types import WellImage, WellMeta
from ome_zarr import NgffMultiscales
from napari_autoscape._converter import convert_leica_composite_to_ngff
from napari_autoscape._utils import _find_in_nested_dict

import zarr
from dataclasses import InitVar, dataclass
from typing import Any, cast
import os
from pathlib import Path
import xmltodict

In [32]:
@dataclass
class NgffHCSPlate:
	"""
    A plate in the HCS specification.
	"""
	images: dict[tuple[str, str], list[NgffMultiscales]]

	def __post_init__(self):

		# sort images first by row (letter) then by column (number)
		self.images = dict(sorted(self.images.items(), key=lambda x: (x[0][1], int(x[0][0]) if x[0][0].isdigit() else x[0][0])))
		self.rows = []
		self.columns = []
		self.wells = []
		for key, value in self.images.items():
			if not key[1] in self.rows:
				self.rows.append(key[1])
			if not key[0] in self.columns:
				self.columns.append(key[0])
			
            # make sure we have a list of NgffMultiscales
			if not all(isinstance(item, NgffMultiscales) for item in value):
				raise TypeError(f"Expected list of NgffMultiscales, got {type(value)} for key {key}")
			
		# convert to ozmp instances
		self.rows = [Row(name=row) for row in self.rows]
		self.columns = [Column(name=column) for column in self.columns]

		# iterate over images again to find correct rowIndex and columnIndex for each well
		for key, value in self.images.items():
			row_index = next(i for i, row in enumerate(self.rows) if row.name == key[1])
			column_index = next(i for i, column in enumerate(self.columns) if column.name == key[0])
			self.wells.append(
				WellInPlate(
					path=f"{key[1]}/{key[0]}",
					rowIndex=row_index,
					columnIndex=column_index
					)
				)
			
		self.plate = Plate(
			rows=self.rows,
			columns=self.columns,
			wells=self.wells,
			acquisitions=[Acquisition(id=1, maximumfieldcount=1)]
		)

	def to_ome_zarr(
        self,
        group: zarr.Group | str,
        storage_options: list[dict[str, Any]] | dict[str, Any] | None = None,
        version: str = "0.5",
        compute: bool = True,
    ) -> list:
	
		import os
		import shutil

		from ome_zarr.format import Format, FormatV04, FormatV05
		from ome_zarr.utils import _recursive_pop_nones
		from ome_zarr.writer import check_group_fmt

		if os.path.exists(str(group)):
			shutil.rmtree(str(group))

		fmt: Format | None = None
		if version == "0.5":
			fmt = FormatV05()
		elif version == "0.4":
			fmt = FormatV04()
		else:
			raise ValueError(f"Unsupported OME-Zarr version: {version}")

		group, fmt = check_group_fmt(group, fmt)

		for key, images_in_well in self.images.items():
			well_group = group.require_group(f"{key[1]}/{key[0]}")
			well_images = []
			for i, image in enumerate(images_in_well):
				image_group = well_group.require_group(f"{i}")
				image.to_ome_zarr(image_group, storage_options=storage_options, version=version, compute=compute)
				well_images.append(WellImage(acquisition=1, path=f"{i}"))

			well_metadata = WellMeta(images=well_images, version=version)

			if version == "0.5":
				well_group.attrs["ome"] = {
					"well": _recursive_pop_nones(well_metadata.model_dump())
				}

		group.attrs["ome"] = {
			"plate": _recursive_pop_nones(self.plate.model_dump())
		}

## Test

In [17]:
root = r'C:\Users\johan\Desktop\local_data\Lightsheet\2026_03_20_10_31_32--Marina_96\TileScan 1'

In [23]:
images_in_plate = {}

for column in os.listdir(root):
	files = os.listdir(os.path.join(root, column))
	for file in files:
		if file.endswith('.ome.tif'):
			row = file.split('.')[0]
			metadata_dict = xmltodict.parse(open(os.path.join(root, column, "Metadata", f"{row}.xlif")).read())
			ngff_img_unique_id = _find_in_nested_dict(metadata_dict, '@Name', row)[0]['@UniqueID']

			ngff_img = convert_leica_composite_to_ngff(os.path.join(root, column, file))
			ngff_img.name = f"{row}_{column}:{ngff_img_unique_id}"
			
			images_in_plate[(row, column)] = [ngff_img]

<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small to downsample further.
<string>:7: UserWarning: Dimension z is too small t

In [33]:
ngff_plate = NgffHCSPlate(images=images_in_plate)

c:\Users\johan\miniforge3\envs\autoscape\Lib\site-packages\pydantic\main.py:250: ValidationWarning: 'version' field not specified in plate metadata, setting version='0.5'
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [34]:
ngff_plate.to_ome_zarr("test_plate.ome.zarr")